# Train PCAP Final
Mặc định tái sử dụng mô hình V3 đã train trên Drive. Không cần tải PCAP hoặc train lại để export. Đặt RETRAIN=True chỉ khi muốn tái tạo thực nghiệm trong thư mục riêng. Các kiểm chứng bổ sung đã chạy được giữ trong evaluation_extra của bản gốc.

## 0. Môi trường và chế độ chạy

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install scapy qiskit
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt
import requests, zipfile, hashlib, json, joblib, re, time, math
from collections import Counter, defaultdict
from scapy.all import PcapReader, IP, TCP, UDP
BASE=Path('/content/drive/MyDrive/lab_MHMM/DoAn/ddos_qae')
assert BASE.exists(), BASE
OUT=BASE/'output/train_v3_pcap'; OUT.mkdir(parents=True,exist_ok=True)
ZIP=BASE/'data/01_synthetic_data.zip'
ROOT=Path('/content/synthetic_v3')
SEED=42; WINDOW=5; LAYERS=1; MAX_IT=60; EPOCHS=60
FEATURES=[f'f{i}' for i in range(1,7)]
CLASSES=['Normal','SYN','HTTP','UDP']
torch.set_num_threads(2)

RETRAIN=False
ORIGINAL_OUT=OUT
if RETRAIN:
    OUT=BASE/"output/train_pcap_final_retrain"
    OUT.mkdir(parents=True,exist_ok=True)


Mounted at /content/drive


## Cell 1 — Tải và giải nén (kiểm tra MD5 Zenodo)

In [2]:
if RETRAIN:
    URL='https://zenodo.org/records/21705919/files/01_synthetic_data.zip?download=1'
    def digest(p,alg='sha256'):
        h=hashlib.new(alg)
        with open(p,'rb') as f:
            for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
        return h.hexdigest()
    if not ZIP.exists():
        tmp=ZIP.with_suffix('.part')
        with requests.get(URL,stream=True,timeout=(30,180)) as r:
            r.raise_for_status()
            with open(tmp,'wb') as f:
                for chunk in r.iter_content(1024*1024): f.write(chunk)
        assert digest(tmp,'md5')=='08df8a1fde02afc3e239c777fe9e9181', 'Checksum sai'
        tmp.replace(ZIP)
    assert digest(ZIP,'md5')=='08df8a1fde02afc3e239c777fe9e9181', 'ZIP sai checksum; kiểm tra lại file'
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        for name in z.namelist():
            assert (ROOT/name).resolve().is_relative_to(ROOT.resolve()), name
        z.extractall(ROOT)
    paths=sorted(p for p in ROOT.rglob('*') if p.suffix.lower() in ('.pcap','.pcapng'))
    print('PCAP:',len(paths)); print(*[str(p.relative_to(ROOT)) for p in paths[:15]],sep='\n')
    assert paths

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 2 — Danh sách nhãn và kiểm tra nguồn
Nhãn đề xuất theo tên file, không suy ra từ feature. Xem toàn bộ bảng, sửa CSV manifest nếu cần rồi đặt CONFIRM_LABELS=True và chạy lại. File cùng SHA256 được gộp, nhãn mâu thuẫn phải sửa. Nhãn theo capture giả định toàn bộ cửa sổ có cùng loại traffic.

In [3]:
if RETRAIN:
    CONFIRM_LABELS=False # đổi True sau khi kiểm tra bảng
    MANIFEST=OUT/'manifest.csv'
    def guess(p):
        tokens=set(re.split(r'[^a-z0-9]+',p.stem.lower()))
        hits=[c for c in CLASSES if c.lower() in tokens or (c=='Normal' and 'benign' in tokens)]
        return hits[0] if len(hits)==1 else 'REVIEW'
    if not MANIFEST.exists():
        manifest=pd.DataFrame([dict(file=str(p.relative_to(ROOT)),label=guess(p),sha256=digest(p)) for p in paths])
        manifest.to_csv(MANIFEST,index=False)
    manifest=pd.read_csv(MANIFEST)
    with pd.option_context('display.max_rows',None): display(manifest[['file','label']])
    print(manifest.label.value_counts()); print('Manifest:',MANIFEST)
    assert CONFIRM_LABELS, 'Kiểm tra nhãn rồi đổi CONFIRM_LABELS=True'
    assert set(manifest.label)==set(CLASSES), 'Sửa nhãn trong manifest.csv, cần đủ 4 lớp'
    assert manifest.file.is_unique
    for row in manifest.itertuples():
        p=(ROOT/row.file).resolve(); assert p.is_relative_to(ROOT.resolve()) and p.is_file() and digest(p)==row.sha256
    assert manifest.groupby('sha256').label.nunique().max()==1
    manifest=manifest.drop_duplicates('sha256').reset_index(drop=True)

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## 3. Feature extractor — dùng chung train và demo

In [4]:
%pip install -q dpkt
# Run in the Train V3 notebook after Cell 2.
# Install dpkt first: %pip install -q dpkt
import dpkt
import math, time, hashlib, os
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from decimal import Decimal
from scapy.all import PcapReader, RawPcapReader, Ether, IP, TCP, UDP

EXTRACTOR_VERSION = 'v3-fast-1'

def scapy_fields(p):
    flags = int(p[TCP].flags) if TCP in p else -1
    src = p[IP].src if IP in p else None
    payload = bytes(p[TCP].payload) if src is not None and TCP in p else b''
    return flags, src, (b'GET' in payload or b'POST' in payload), UDP in p

def bucket():
    return [0, 0, 0, Counter(), Counter(), 0]

def add(buckets, t, start, window, fields):
    flags, src, http, udp = fields
    a = buckets[int((t-start)/window)]
    a[0] += 1
    a[1] += int(flags == 2)
    a[2] += int(flags == 18)
    if src is not None:
        a[3][src] += 1
        if http:
            a[4][src] += 1
    a[5] += int(udp)

def finish(buckets, start, end, window):
    rows = []
    if math.isfinite(start) and math.isfinite(end):
        full = int((end-start)/window)
        for i, a in sorted(buckets.items()):
            if i >= full:
                continue
            total, syn, sa, ips, http, udp = a
            entropy = -sum((c/total)*math.log2(c/total) for c in ips.values())
            rows.append([i*window, total, syn/max(sa,1), len(ips),
                         max(http.values(), default=0), entropy, udp])
    return pd.DataFrame(rows, columns=['window_start_s'] + FEATURES)

def extract_reference(path, window=WINDOW):
    # Reference: corrected Scapy implementation (two passes).
    start, end = float('inf'), float('-inf')
    with PcapReader(str(path)) as reader:
        for p in reader:
            t = float(p.time)
            start, end = min(start,t), max(end,t)
    buckets = defaultdict(bucket)
    with PcapReader(str(path)) as reader:
        for p in reader:
            add(buckets, float(p.time), start, window, scapy_fields(p))
    return finish(buckets, start, end, window)

def fast_fields(raw):
    # Fast path only for ordinary, complete Ethernet/IPv4 TCP or UDP.
    # Other packet structures are decoded with Scapy rather than discarded.
    try:
        if len(raw) < 34 or raw[12:14] != b'\x08\x00':
            raise ValueError('Non-standard Ethernet/IPv4')
        ip = dpkt.ip.IP(raw[14:])
        if (ip.v != 4 or ip.hl != 5 or ip.len != len(raw)-14
                or ip.off & 0x3fff):
            raise ValueError('Options, fragments, padding or truncation')
        src = '.'.join(map(str, ip.src))
        if ip.p == 6 and isinstance(ip.data, dpkt.tcp.TCP):
            tcp = ip.data
            if tcp.off != 5 or len(ip.data) != ip.len-20:
                raise ValueError('TCP options or unusual length')
            payload = bytes(tcp.data)
            return tcp.flags, src, (b'GET' in payload or b'POST' in payload), False
        if ip.p == 17 and isinstance(ip.data, dpkt.udp.UDP):
            if ip.data.ulen != ip.len-20 or len(ip.data) != ip.len-20:
                raise ValueError('Unusual UDP length')
            return -1, src, False, True
    except (dpkt.UnpackError, ValueError, TypeError, IndexError):
        pass
    return scapy_fields(Ether(raw))

def extract_pcap(path, window=WINDOW):
    if window <= 0:
        raise ValueError('Window must be positive')
    records = []
    start, end = float('inf'), float('-inf')
    # RawPcapReader supplies timestamps without decoding every packet in Scapy.
    with RawPcapReader(str(path)) as reader:
        if getattr(reader,'linktype',None) != 1 or not hasattr(reader,'nano'):
            return extract_reference(path, window)
        resolution = Decimal(10**9 if reader.nano else 10**6)
        for raw, meta in reader:
            # Preserve Scapy's decimal-to-float timestamp convention.
            t = float(Decimal(meta.sec) + Decimal(meta.usec)/resolution)
            start, end = min(start,t), max(end,t)
            records.append((t, fast_fields(raw)))
    buckets = defaultdict(bucket)
    for t, fields in records:
        add(buckets, t, start, window, fields)
    return finish(buckets, start, end, window)



In [5]:
if RETRAIN:
    # Verify the fast extractor on one smallest capture per class.
    # Set VERIFY_ALL=True to compare every capture (slower).
    VERIFY_ALL = False
    checks = manifest.copy()
    checks['_bytes'] = checks.file.map(lambda f: (ROOT/f).stat().st_size)
    if not VERIFY_ALL:
        checks = checks.sort_values('_bytes').groupby('label',sort=False).head(1)
    for row in checks.itertuples():
        t = time.perf_counter(); fast = extract_pcap(ROOT/row.file); tf = time.perf_counter()-t
        t = time.perf_counter(); ref = extract_reference(ROOT/row.file); ts = time.perf_counter()-t
        pd.testing.assert_frame_equal(fast, ref, check_exact=False, rtol=1e-10, atol=1e-10)
        print(f'Verified {row.label}: {row.file} | fast {tf:.2f}s | Scapy {ts:.2f}s', flush=True)
    print('Verification passed. No claim of equivalence on unverified captures.')

    # Cache per capture; content hash + window + extractor version prevent stale reuse.
    CACHE = OUT/'feature_cache_fast_v1'
    CACHE.mkdir(parents=True, exist_ok=True)
    frames, failures = [], []
    t0 = time.perf_counter(); processed_bytes = 0; total_bytes = sum((ROOT/f).stat().st_size for f in manifest.file)
    for idx, row in enumerate(manifest.itertuples(), 1):
        try:
            path = ROOT/row.file
            key = hashlib.sha256(f'{row.sha256}|{WINDOW}|{EXTRACTOR_VERSION}'.encode()).hexdigest()
            cp = CACHE/f'{key}.csv'
            t = time.perf_counter()
            if cp.exists():
                d = pd.read_csv(cp); status = 'CACHE'
            else:
                d = extract_pcap(path); status = 'OK'
                if d.empty: raise ValueError('Không có cửa sổ đầy đủ')
                assert np.isfinite(d[FEATURES]).all().all()
                tmp = cp.with_suffix('.tmp'); d.to_csv(tmp,index=False); os.replace(tmp,cp)
            assert list(d.columns)==['window_start_s']+FEATURES and not d.empty
            assert np.isfinite(d[FEATURES]).all().all()
            d['file'], d['label'] = row.file, row.label
            frames.append(d); processed_bytes += path.stat().st_size
            elapsed = time.perf_counter()-t0
            eta = elapsed*(total_bytes-processed_bytes)/max(processed_bytes,1)/60
            print(f'[{idx}/{len(manifest)}] {row.label} | {len(d)} windows | {status} | {time.perf_counter()-t:.1f}s | ETA ~{eta:.1f} min',flush=True)
        except Exception as e:
            failures.append((row.file,str(e))); print(f'ERROR: {row.file}: {e}',flush=True)
    print('Lỗi:', failures)
    assert not failures, 'Kiểm tra file lỗi trước khi train'
    data = pd.concat(frames, ignore_index=True)
    assert np.isfinite(data[FEATURES]).all().all()
    assert data.file.nunique()==len(manifest)
    data.to_csv(OUT/'features.csv',index=False)
    print('Tổng số mẫu:',len(data),'| thời gian:',round((time.perf_counter()-t0)/60,2),'phút')
    display(data.head()); display(data.groupby('label').agg(windows=('file','size'),files=('file','nunique')))

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 4 — Split theo file PCAP, không theo dòng
60/20/20 gần đúng, phân tầng theo nhãn. QAE fit trên file normal thuộc train. Scaler dùng train; validation chọn ngưỡng. Không cân bằng test. Báo cáo cả số dòng trùng giữa các tập vì feature tổng hợp có thể giống nhau dù capture khác.

In [6]:
if RETRAIN:
    files=data[['file','label']].drop_duplicates()
    assert files.label.value_counts().min()>=5, 'Cần ít nhất 5 capture mỗi lớp'
    tr,rest=train_test_split(files,test_size=.4,stratify=files.label,random_state=SEED)
    va,te=train_test_split(rest,test_size=.5,stratify=rest.label,random_state=SEED)
    parts={'train':tr,'val':va,'test':te}
    for k,v in parts.items():
        assert set(v.label)==set(CLASSES)
        print(k,v.label.value_counts().to_dict())
    assert not (set(tr.file)&set(te.file) or set(tr.file)&set(va.file) or set(va.file)&set(te.file))
    split=pd.concat([v.assign(split=k) for k,v in parts.items()]); split.to_csv(OUT/'split.csv',index=False)
    D={k:data[data.file.isin(v.file)].reset_index(drop=True) for k,v in parts.items()}
    for a,b in [('train','val'),('train','test')]:
        print('Feature trùng',a,b,len(set(map(tuple,D[a][FEATURES].values))&set(map(tuple,D[b][FEATURES].values))))
    scaler=MinMaxScaler(clip=True).fit(np.log1p(D['train'][FEATURES].to_numpy()))
    def transform(d,sc=scaler): return sc.transform(np.log1p(d[FEATURES].to_numpy()))
    A={k:transform(d) for k,d in D.items()}
    y={k:d.label.map({c:i for i,c in enumerate(CLASSES)}).to_numpy() for k,d in D.items()}
    print({k:a.shape for k,a in A.items()})

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 5 — QAE, mạch 3 qubit; kiểm tra SWAP test

In [7]:
def amplitudes(A):
    Z=np.pad(A,((0,0),(0,2))).astype(float); norms=np.linalg.norm(Z,axis=1)
    Z[norms==0,0]=1; return Z/np.linalg.norm(Z,axis=1,keepdims=True)
def encoder(theta,layers):
    qc=QuantumCircuit(3); t=np.asarray(theta).reshape(layers,3,3)
    for layer in t:
        for i,(a,b,c) in enumerate(layer): qc.rx(a,i); qc.ry(b,i); qc.rz(c,i)
        qc.cx(0,1); qc.cx(1,2); qc.cx(2,0)
    return qc

def qae_fit(A,seed,layers=LAYERS):
    Z=amplitudes(A); rho=Z.T@Z/len(Z)
    def loss(t):
        U=Operator(encoder(t,layers)).data; R=U@rho@U.conj().T
        return float(1-np.trace(R[:4,:4]).real)
    t0=np.random.default_rng(seed).normal(0,.2,9*layers)
    r=minimize(loss,t0,method='L-BFGS-B',options={'maxiter':MAX_IT,'ftol':1e-9})
    return r.x,dict(loss=float(r.fun),iterations=int(r.nit),converged=bool(r.success),message=str(r.message))

def qae_features(A,t,layers=LAYERS):
    Z=amplitudes(A)@Operator(encoder(t,layers)).data.T
    p=np.abs(Z)**2
    score=p[:,4:].sum(1)
    latent=p[:,:4]+p[:,4:] # marginal probabilities of q0,q1
    return np.column_stack([score,latent])




In [8]:
if RETRAIN:
    theta,info=qae_fit(A['train'][y['train']==0],SEED); print(info)
    qc=QuantumCircuit(5); qc.initialize(amplitudes(A['train'][:1])[0],[0,1,2]); qc.compose(encoder(theta,LAYERS),[0,1,2],inplace=True); qc.h(4); qc.cswap(4,2,3); qc.h(4)
    p0=Statevector.from_instruction(qc).probabilities([4])[0]
    assert np.isclose(qae_features(A['train'][:1],theta)[0,0],2*(1-p0),atol=1e-8)
    display(qc.draw(output='text'))
    Q={k:qae_features(a,theta) for k,a in A.items()}
    threshold=float(np.quantile(Q['val'][y['val']==0,0],.9))
    print('QAE threshold',threshold)

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 6 — QNN biến phân, QAE + QNN
Mạch Ry/CX 3 qubit mô phỏng statevector khả vi bằng PyTorch, đầu ra xác suất đo và linear head. QAE được fit trước và đóng băng. Class weights chỉ lấy từ train. Cùng train/test cho các classifier; đây là pipeline hai giai đoạn, không joint training.

In [9]:
def ry(state,angle,q):
    # state[:, i], bit q theo thứ tự little endian của Qiskit
    idx=torch.arange(8,device=state.device); partner=idx^(1<<q)
    sign=torch.where((idx&(1<<q))==0,-1.,1.)
    a=angle.reshape(-1,1)/2
    return torch.cos(a)*state+torch.sin(a)*state[:,partner]*sign

def cx(state,c,t):
    idx=torch.arange(8,device=state.device); return state[:,idx^(((idx>>c)&1)<<t)]

def ring(s):
    for c,t in [(0,1),(1,2),(2,0)]: s=cx(s,c,t)
    return s

class VQNN(nn.Module):
    def __init__(self,nclass,extra=0):
        super().__init__(); self.theta=nn.Parameter(torch.randn(2,3)*.2); self.head=nn.Linear(8+extra,nclass)
    def forward(self,x):
        s=torch.zeros((len(x),8),device=x.device); s[:,0]=1
        for block in range(2):
            for q in range(3): s=ry(s,x[:,block*3+q]*np.pi,q)
            s=ring(s)
        for layer in self.theta:
            for q in range(3): s=ry(s,layer[q],q)
            s=ring(s)
        return self.head(torch.cat([s.square(),x[:,6:]],dim=1))

class CNN(nn.Module):
    def __init__(self,nclass):
        super().__init__(); self.net=nn.Sequential(nn.Conv1d(1,16,3,padding=1),nn.ReLU(),nn.Conv1d(16,16,3,padding=1),nn.ReLU(),nn.Flatten(),nn.Linear(96,nclass))
    def forward(self,x): return self.net(x.unsqueeze(1))

def train_torch(model,A,y,B,seed):
    opt=torch.optim.Adam(model.parameters(),lr=.01,weight_decay=1e-4)
    weights=torch.tensor(len(y)/(len(np.unique(y))*np.bincount(y)),dtype=torch.float32)
    loss=nn.CrossEntropyLoss(weight=weights)
    dl=DataLoader(TensorDataset(torch.tensor(A,dtype=torch.float32),torch.tensor(y,dtype=torch.long)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(seed))
    model.train()
    for epoch in range(EPOCHS):
        for x,t in dl:
            opt.zero_grad(); l=loss(model(x),t); l.backward(); opt.step()
    model.eval()
    with torch.no_grad(): p=model(torch.tensor(B,dtype=torch.float32)).softmax(1).numpy()
    return model,p



In [10]:
if RETRAIN:
    # kiểm tra cổng Ry/CX với Qiskit
    s=torch.zeros(1,8); s[:,0]=1; s=ry(s,torch.tensor(.7),1); s=cx(s,1,2)
    qc=QuantumCircuit(3); qc.ry(.7,1); qc.cx(1,2)
    assert np.allclose(s.numpy()[0],Statevector.from_instruction(qc).data,atol=1e-6)
    print('QNN gate check OK')
    H={k:np.column_stack([A[k],Q[k]]) for k in A}
    models={}; probs={}; times={}
    for name,inputs,extra in [('QNN',A,0),('QAE+QNN',H,5)]:
        torch.manual_seed(SEED); t=time.perf_counter()
        models[name],probs[name]=train_torch(VQNN(4,extra),inputs['train'],y['train'],inputs['test'],SEED)
        times[name]=time.perf_counter()-t
        print(name,'seconds',times[name])
    for name,m in [('RF',RandomForestClassifier(n_estimators=150,class_weight='balanced',random_state=SEED)),('MLP',MLPClassifier(hidden_layer_sizes=(32,16),max_iter=500,random_state=SEED))]:
        t=time.perf_counter(); m.fit(A['train'],y['train']); probs[name]=m.predict_proba(A['test']); times[name]=time.perf_counter()-t

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 7 — Kết quả trên PCAP test chưa dùng để train
Binary từ multiclass: cộng xác suất 3 lớp attack, ngưỡng 0.5 cố định. QAE riêng dùng ngưỡng validation. Kết quả theo cửa sổ, không phải từng gói. Không chọn file demo dựa trên dự đoán đúng.

In [11]:
if RETRAIN:
    rows=[]
    for name,p in probs.items():
        pred=p.argmax(1); binary=(1-p[:,0]>=.5).astype(int); truth=(y['test']!=0).astype(int)
        rows.append(dict(model=name,accuracy=accuracy_score(y['test'],pred),macro_f1=f1_score(y['test'],pred,average='macro'),binary_f1=f1_score(truth,binary),binary_auc=roc_auc_score(truth,1-p[:,0]),seconds=times[name]))
        print(name); print(classification_report(y['test'],pred,labels=range(4),target_names=CLASSES,zero_division=0))
    metrics=pd.DataFrame(rows); display(metrics); metrics.to_csv(OUT/'metrics.csv',index=False)
    print('QAE binary AUC:',roc_auc_score(y['test']!=0,Q['test'][:,0]))
    print(classification_report(y['test']!=0,Q['test'][:,0]>threshold,zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y['test'],probs['QAE+QNN'].argmax(1),labels=range(4),display_labels=CLASSES,cmap='Blues'); plt.tight_layout(); plt.savefig(OUT/'confusion.png',dpi=160); plt.show()

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## Cell 8 — Lưu mô hình và metadata
Bundle chỉ load file tự tạo, không load joblib từ nguồn lạ. Lưu danh sách train/val/test và SHA256 để truy vết.

In [12]:
if RETRAIN:
    joblib.dump(scaler,OUT/'scaler.joblib')
    np.save(OUT/'qae_theta.npy',theta)
    torch.save(models['QAE+QNN'].state_dict(),OUT/'hybrid.pt')
    (OUT/'config.json').write_text(json.dumps(dict(window=WINDOW,layers=LAYERS,classes=CLASSES,features=FEATURES,threshold=threshold,seed=SEED,epochs=EPOCHS,extractor='algorithm2_adapted_nonempty_full_v1',qae=info),indent=2))
    print('Saved',OUT)

else:
    print("Reuse existing V3 artifacts")

Reuse existing V3 artifacts


## 9. Load mô hình và inference
Có thể kiểm thử PCAP mới bằng đường dẫn trên Colab. Mặc định đánh giá lại feature test đã lưu, không đọc lại PCAP.

In [13]:
def transform(d,sc):
    return sc.transform(np.log1p(d[FEATURES].to_numpy()))
cfg=json.loads((OUT/'config.json').read_text()); assert cfg['features']==FEATURES and cfg['classes']==CLASSES
saved_scaler=joblib.load(OUT/'scaler.joblib'); saved_theta=np.load(OUT/'qae_theta.npy')
saved_model=VQNN(4,5); saved_model.load_state_dict(torch.load(OUT/'hybrid.pt',map_location='cpu',weights_only=True)); saved_model.eval()
def predict_pcap(path):
    d=extract_pcap(path,cfg['window'])
    if d.empty: raise ValueError('Không có cửa sổ đầy đủ')
    a=transform(d,saved_scaler); q=qae_features(a,saved_theta,cfg['layers'])
    with torch.no_grad(): p=saved_model(torch.tensor(np.column_stack([a,q]),dtype=torch.float32)).softmax(1).numpy()
    d['anomaly_score']=q[:,0]; d['qae_alert']=q[:,0]>cfg['threshold']; d['prediction']=np.array(cfg['classes'])[p.argmax(1)]; d['confidence']=p.max(1); d['attack_probability']=1-p[:,0]
    return d


In [14]:
data=pd.read_csv(OUT/'features.csv')
sp=pd.read_csv(OUT/'split.csv')
test=data[data.file.isin(sp.loc[sp.split=='test','file'])].copy()
a=transform(test,saved_scaler); q=qae_features(a,saved_theta,cfg['layers'])
with torch.no_grad(): prob=saved_model(torch.tensor(np.column_stack([a,q]),dtype=torch.float32)).softmax(1).numpy()
pred=np.array(cfg['classes'])[prob.argmax(1)]
print(classification_report(test.label,pred,labels=cfg['classes'],zero_division=0))
DEMO_FILE=None # ví dụ '/content/capture.pcap'
if DEMO_FILE:
    display(predict_pcap(Path(DEMO_FILE)))
extra=OUT/'evaluation_extra'
if extra.exists():
    for f in ['leakage_check.csv','stability_summary.csv','unseen_intensity_results.csv','novel_metrics.csv']:
        if (extra/f).exists(): print(f); display(pd.read_csv(extra/f))

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        40
         SYN       1.00      1.00      1.00        18
        HTTP       1.00      1.00      1.00        15
         UDP       1.00      1.00      1.00        10

    accuracy                           1.00        83
   macro avg       1.00      1.00      1.00        83
weighted avg       1.00      1.00      1.00        83

leakage_check.csv


,split_A,split_B,duplicate_files,duplicate_sha256,duplicate_features
0,train,val,0,0,17
1,train,test,0,0,18
2,val,test,0,0,11


stability_summary.csv


,Unnamed: 0,accuracy,accuracy.1,macro_f1,macro_f1.1,binary_f1,binary_f1.1,binary_auc,binary_auc.1,seconds,seconds.1
0,NaN,mean,std,mean,std,mean,std,mean,std,mean,std
1,model,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MLP,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.27555633666634094,0.011084387016850654
3,QAE,0.9320615796519411,0.0385250963764013,NaN,NaN,0.9354082954570405,0.03317649192172637,1.0,0.0,0.27028208100000484,0.026445827583962394
4,QAE+QNN,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,2.1242239680001753,0.6719929412495149
5,QNN,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,2.0158356863333515,0.8745043069021448
6,RF,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.24502572000013365,0.027827777893787597


unseen_intensity_results.csv


,experiment,seed,model,accuracy,macro_f1,binary_f1,binary_auc,seconds,test_samples
0,Unseen HIGH intensity,42,QAE,0.922414,NaN,0.940397,1.0,0.342554,116
1,Unseen HIGH intensity,42,QNN,1.000000,1.0,1.000000,1.0,1.289771,116
2,Unseen HIGH intensity,42,QAE+QNN,1.000000,1.0,1.000000,1.0,1.564065,116
3,Unseen HIGH intensity,42,RF,1.000000,1.0,1.000000,1.0,0.226844,116
4,Unseen HIGH intensity,42,MLP,1.000000,1.0,1.000000,1.0,0.254119,116


novel_metrics.csv


,model,test_samples,correct,accuracy,macro_f1_present_classes
0,QAE+QNN,50,50,1.0,1.0


## 10. Export về laptop D:\QAE_QNN
ZIP chỉ chứa model và kết quả; không có PCAP hoặc giao diện. Chỉ load model do bạn tạo.

In [15]:
from google.colab import files as colab_files
from zipfile import ZipFile, ZIP_DEFLATED
from importlib.metadata import version
bundle=Path('/content/Train_PCAP_Export.zip')
with ZipFile(bundle,'w',ZIP_DEFLATED) as z:
    for name in ['hybrid.pt','scaler.joblib','qae_theta.npy','config.json']:
        z.write(OUT/name,'models/'+name)
    for name in ['metrics.csv','confusion.png','features.csv','split.csv','manifest.csv']:
        if (OUT/name).exists(): z.write(OUT/name,'results/pcap/'+name)
    for f in OUT.glob('*_prediction.csv'):
        z.write(f,'results/pcap/predictions/'+f.name)
    for f in (OUT/'evaluation_extra').rglob('*'):
        if f.is_file(): z.write(f,'results/pcap/evaluation_extra/'+str(f.relative_to(OUT/'evaluation_extra')))
    z.writestr('models/environment.json',json.dumps({k:version(k) for k in ['torch','scikit-learn','numpy','pandas','qiskit','scapy','dpkt','joblib']},indent=2))
print(bundle, bundle.stat().st_size)
colab_files.download(str(bundle))

/content/Train_PCAP_Export.zip 327408


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>